# Notebook 10 — Perbandingan K-Means vs HDBSCAN (UMAP embedding identik)

Eksperimen perbandingan **adil** antara dua algoritma clustering pada **array UMAP 30-dim yang sama persis** dengan NB09.

- **Skenario A (acuan, dari NB09):** UMAP -> **HDBSCAN** -> Silhouette = 0.9041, 95 cluster, coverage 99.3%, noise 0.7%
- **Skenario B (eksperimen ini):** UMAP -> **K-Means** -> cari `k` optimal (coarse -> fine), ukur Silhouette + DBI

Tujuan: membuktikan secara empiris bahwa HDBSCAN mengungguli K-Means pada dataset yang sama, sekaligus mendokumentasikan kelemahan K-Means (wajib tentukan `k`, tidak ada noise handling, coverage 100% dipaksakan).

> **Cara pakai:** jalankan sel di bawah ini di **Google Colab** (Google Drive akan di-mount). Sel ini *tidak* meregenerasi UMAP — ia memuat `reduced_embeddings` 30-dim yang sudah tersimpan dari NB09 agar perbandingan adil. Jangan ubah file logic apapun (`pipeline.py`, `clustering.py`, `config.py`, `face_extractor.py`, `app.py`).

In [ ]:
# =============================================================================
# NB10 — K-Means vs HDBSCAN pada UMAP embedding identik (dari NB09)
# Jalankan di Google Colab. TIDAK meregenerasi UMAP. TIDAK menyentuh file logic.
# =============================================================================

# ── 1. Imports + konstanta ───────────────────────────────────────────────────
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# random_state dikunci di mana-mana demi reproducibility (KMeans + sampling silhouette)
RANDOM_STATE = 42

PKL_PATH    = '/content/drive/MyDrive/OTW S.KOM/Results/notebook9_umap_results.pkl'
RESULTS_DIR = '/content/drive/MyDrive/OTW S.KOM/Results/'

# Acuan HDBSCAN dari NB09 (best_by_coverage) — dipakai untuk gating sanity check.
HDBSCAN_SILHOUETTE_REF = 0.9041   # nilai yang HARUS kita reproduksi
HDBSCAN_N_CLUSTERS_REF = 95
HDBSCAN_COVERAGE_REF   = 0.993
HDBSCAN_NOISE_REF      = 0.007
SIL_TOLERANCE          = 0.01     # toleransi kecocokan silhouette acuan

# ── 2. Mount Google Drive + load pkl ─────────────────────────────────────────
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Google Drive sudah ter-mount.")

# Jika pkl NB09 tidak ada: berhenti dengan pesan eksplisit. JANGAN regenerate UMAP
# diam-diam (UMAP stochastic; regenerate butuh random_state yang sama dgn NB09 untuk
# bisa mereproduksi angka 0.9041 — itu di luar scope eksperimen ini).
if not os.path.exists(PKL_PATH):
    raise FileNotFoundError(
        f"\nFile NB09 TIDAK ditemukan: {PKL_PATH}\n"
        "Eksperimen ini WAJIB memakai reduced_embeddings 30-dim yang tersimpan dari NB09 "
        "agar K-Means & HDBSCAN berjalan di array yang identik.\n"
        "JANGAN regenerate UMAP secara diam-diam: hasilnya stochastic dan butuh "
        "random_state UMAP yang sama persis dgn NB09 untuk mereproduksi Silhouette 0.9041.\n"
        "Pastikan file pkl tersedia di Drive, lalu jalankan ulang sel ini."
    )

with open(PKL_PATH, 'rb') as f:
    nb9 = pickle.load(f)

# ── 3. Inspeksi + seleksi sumber (anti-asumsi nama key) ──────────────────────
# Pkl NB09 menyimpan DUA kandidat: best_by_coverage & best_by_silhouette. Keduanya
# config berbeda; hanya best_by_coverage yang menyimpan array+label DAN ber-silhouette
# 0.9041 (acuan brief). Maka kita PILIH entri berdasarkan kecocokan 0.9041 + ketersediaan
# array, bukan "silhouette tertinggi" (yang justru menunjuk best_by_silhouette 0.9098
# tanpa array). Tetap defensif terhadap struktur flat.
print("Top-level keys pkl:", list(nb9.keys()))
print("-" * 70)

def _entry_summary(name, e):
    return (f"  {name:<20} sil={e.get('silhouette')} "
            f"cov={e.get('coverage_rate')} "
            f"has_arr={'reduced_embeddings' in e} "
            f"has_lbl={'labels' in e}")

# Kumpulkan kandidat: entri nested yang dikenal + kemungkinan struktur flat
candidates = {}
for key in ('best_by_coverage', 'best_by_silhouette'):
    if isinstance(nb9.get(key), dict):
        candidates[key] = nb9[key]
# Fallback flat: seluruh pkl diperlakukan sbg satu kandidat
if 'reduced_embeddings' in nb9 and ('labels' in nb9 or 'labels_hdbscan' in nb9):
    flat = dict(nb9)
    if 'labels' not in flat and 'labels_hdbscan' in flat:
        flat['labels'] = flat['labels_hdbscan']
    candidates['<flat-root>'] = flat

print("Kandidat sumber yang ditemukan:")
for name, e in candidates.items():
    print(_entry_summary(name, e))
print("-" * 70)

# Pilih entri yang: (a) punya array + label, DAN (b) silhouette ≈ 0.9041
chosen_name, chosen = None, None
for name, e in candidates.items():
    has_arr = 'reduced_embeddings' in e
    has_lbl = 'labels' in e
    sil     = e.get('silhouette')
    if has_arr and has_lbl and sil is not None and abs(sil - HDBSCAN_SILHOUETTE_REF) <= SIL_TOLERANCE:
        chosen_name, chosen = name, e
        break

if chosen is None:
    raise ValueError(
        "Tidak ada entri di pkl yang memenuhi syarat: punya reduced_embeddings + labels "
        f"DAN silhouette ≈ {HDBSCAN_SILHOUETTE_REF} (±{SIL_TOLERANCE}).\n"
        "Kemungkinan struktur pkl berbeda dari yang diharapkan. Periksa ringkasan kandidat "
        "di atas. JANGAN regenerate UMAP — verifikasi sumber data dulu."
    )

print(f"Sumber terpilih: '{chosen_name}' (silhouette tersimpan = {chosen.get('silhouette')})")

umap_embeddings = np.asarray(chosen['reduced_embeddings'])
labels_hdbscan  = np.asarray(chosen['labels'])
dbi_stored      = chosen.get('dbi')   # hanya untuk cross-check, bukan sumber utama

print(f"umap_embeddings.shape = {umap_embeddings.shape}")
assert umap_embeddings.ndim == 2 and umap_embeddings.shape[1] == 30, \
    f"Diharapkan array UMAP 30-dim, didapat {umap_embeddings.shape}"
assert umap_embeddings.shape[0] == labels_hdbscan.shape[0], \
    "Jumlah baris embedding != jumlah label HDBSCAN."

# ── 4. Sanity check HDBSCAN (gating) + DBI dihitung ulang pada non-noise ──────
# Silhouette HDBSCAN dihitung HANYA pada titik non-noise (label != -1), persis seperti
# NB09. Kalau tidak match ~0.9041 -> array yang di-load kemungkinan salah -> STOP.
mask_nonnoise = labels_hdbscan != -1
n_total       = len(labels_hdbscan)
n_nonnoise    = int(mask_nonnoise.sum())

emb_nn = umap_embeddings[mask_nonnoise]
lbl_nn = labels_hdbscan[mask_nonnoise]

sil_hdbscan = silhouette_score(emb_nn, lbl_nn)
# DBI HDBSCAN dihitung ULANG di sini pada non-noise -> metodologi identik dgn K-Means
# (yang otomatis tanpa noise). dbi tersimpan hanya dijadikan cross-check.
dbi_hdbscan = davies_bouldin_score(emb_nn, lbl_nn)

n_clusters_hdbscan = len(set(lbl_nn.tolist()))
coverage_hdbscan   = n_nonnoise / n_total
noise_hdbscan      = 1.0 - coverage_hdbscan

print("\n=== SANITY CHECK HDBSCAN (non-noise) ===")
print(f"  n_total          : {n_total}")
print(f"  n_non-noise      : {n_nonnoise}")
print(f"  n_clusters       : {n_clusters_hdbscan}")
print(f"  Silhouette       : {sil_hdbscan:.4f}  (acuan {HDBSCAN_SILHOUETTE_REF})")
print(f"  DBI (recompute)  : {dbi_hdbscan:.4f}  (tersimpan: {dbi_stored})")
print(f"  Coverage         : {coverage_hdbscan:.1%}")
print(f"  Noise            : {noise_hdbscan:.1%}")

if abs(sil_hdbscan - HDBSCAN_SILHOUETTE_REF) > SIL_TOLERANCE:
    raise ValueError(
        f"PERINGATAN: Silhouette HDBSCAN ({sil_hdbscan:.4f}) TIDAK cocok dengan acuan "
        f"{HDBSCAN_SILHOUETTE_REF} (±{SIL_TOLERANCE}). Array/label yang di-load kemungkinan "
        "salah. STOP sebelum K-Means — verifikasi sumber data dulu."
    )
print("  -> OK, cocok dengan acuan. Lanjut ke K-Means.\n")

# Untuk K-Means, fairness: gunakan SELURUH titik UMAP yang sama (K-Means tak punya
# konsep noise -> semua titik di-assign). Ini sengaja, untuk menyorot keterbatasannya.
X = umap_embeddings
sample_size = min(1000, len(X))   # silhouette O(n^2) -> sampling utk pencarian k

# ── 5. Coarse search K-Means: k in [70..130] step 10 ─────────────────────────
coarse_ks = list(range(70, 131, 10))
coarse_scores = []
print("=== COARSE SEARCH K-MEANS ===")
print(f"  {'k':>4} | {'silhouette (sample)':>20}")
print(f"  {'-'*4} | {'-'*20}")
for k in coarse_ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lbl = km.fit_predict(X)
    sil = silhouette_score(X, lbl, sample_size=sample_size, random_state=RANDOM_STATE)
    coarse_scores.append(sil)
    print(f"  {k:>4} | {sil:>20.4f}")

k_peak = coarse_ks[int(np.argmax(coarse_scores))]
print(f"\n  k_peak (coarse) = {k_peak}")

# Guard tepi: kalau puncak jatuh di ujung range, optimum mungkin di luar 70-130.
if k_peak in (coarse_ks[0], coarse_ks[-1]):
    print(f"  PERINGATAN: k_peak={k_peak} berada di TEPI range coarse [{coarse_ks[0]}..{coarse_ks[-1]}].")
    print( "  Optimum sebenarnya mungkin di luar range ini — pertimbangkan melebarkan coarse search.")

# ── 6. Fine search K-Means: ±5 di sekitar k_peak ─────────────────────────────
fine_ks = list(range(max(2, k_peak - 5), k_peak + 6))
fine_scores = []
print("\n=== FINE SEARCH K-MEANS ===")
print(f"  {'k':>4} | {'silhouette (sample)':>20}")
print(f"  {'-'*4} | {'-'*20}")
for k in fine_ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lbl = km.fit_predict(X)
    sil = silhouette_score(X, lbl, sample_size=sample_size, random_state=RANDOM_STATE)
    fine_scores.append(sil)
    print(f"  {k:>4} | {sil:>20.4f}")

k_optimal = fine_ks[int(np.argmax(fine_scores))]
print(f"\n  k_optimal (fine) = {k_optimal}")

# ── 7. Evaluasi final K-Means di k_optimal pada FULL data (tanpa sampling) ────
km_final   = KMeans(n_clusters=k_optimal, random_state=RANDOM_STATE, n_init=10)
lbl_kmeans = km_final.fit_predict(X)

sil_kmeans = silhouette_score(X, lbl_kmeans)          # full data -> angka final akurat
dbi_kmeans = davies_bouldin_score(X, lbl_kmeans)      # makin rendah makin baik
coverage_kmeans = 1.0   # K-Means meng-assign SEMUA titik (keterbatasan, bukan keunggulan)
noise_kmeans    = 0.0   # K-Means tidak punya noise handling

print("\n=== EVALUASI FINAL K-MEANS (full data) ===")
print(f"  k_optimal   : {k_optimal}")
print(f"  Silhouette  : {sil_kmeans:.4f}")
print(f"  DBI         : {dbi_kmeans:.4f}")

# ── 8. Tabel perbandingan + simpan CSV ───────────────────────────────────────
comparison = pd.DataFrame([
    {'Metrik': 'Jumlah cluster',   'HDBSCAN': n_clusters_hdbscan,        'K-Means': k_optimal},
    {'Metrik': 'Silhouette [up]',  'HDBSCAN': round(sil_hdbscan, 4),     'K-Means': round(sil_kmeans, 4)},
    {'Metrik': 'DBI [down]',       'HDBSCAN': round(dbi_hdbscan, 4),     'K-Means': round(dbi_kmeans, 4)},
    {'Metrik': 'Coverage',         'HDBSCAN': f"{coverage_hdbscan:.1%}", 'K-Means': "100% (dipaksa)"},
    {'Metrik': 'Noise terdeteksi', 'HDBSCAN': f"{noise_hdbscan:.1%}",    'K-Means': "0% (tidak mampu)"},
    {'Metrik': 'Perlu tentukan k?','HDBSCAN': "Tidak (otomatis)",        'K-Means': "Ya (coarse->fine)"},
])

print("\n" + "=" * 70)
print("TABEL PERBANDINGAN: HDBSCAN vs K-Means (UMAP 30-dim identik)")
print("=" * 70)
print(comparison.to_string(index=False))
print("=" * 70)

os.makedirs(RESULTS_DIR, exist_ok=True)
csv_path = os.path.join(RESULTS_DIR, 'nb10_kmeans_vs_hdbscan.csv')
comparison.to_csv(csv_path, index=False)
print(f"\nTabel disimpan: {csv_path}")

# ── 9. Grafik Silhouette-vs-k (coarse + fine) -> PNG untuk lampiran thesis ────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(coarse_ks, coarse_scores, 'o-', color='#2563eb')
axes[0].axvline(k_peak, ls='--', color='gray', label=f'k_peak={k_peak}')
axes[0].set_title('Coarse search: Silhouette vs k')
axes[0].set_xlabel('k (jumlah cluster)'); axes[0].set_ylabel('Silhouette (sample)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(fine_ks, fine_scores, 'o-', color='#16a34a')
axes[1].axvline(k_optimal, ls='--', color='gray', label=f'k_optimal={k_optimal}')
axes[1].axhline(HDBSCAN_SILHOUETTE_REF, ls=':', color='red',
                label=f'HDBSCAN={HDBSCAN_SILHOUETTE_REF}')
axes[1].set_title('Fine search: Silhouette vs k')
axes[1].set_xlabel('k (jumlah cluster)'); axes[1].set_ylabel('Silhouette (sample)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
png_path = os.path.join(RESULTS_DIR, 'nb10_silhouette_vs_k.png')
plt.savefig(png_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Grafik disimpan: {png_path}")

# ── Kesimpulan ringkas (untuk narasi sidang) ─────────────────────────────────
print("\n" + "=" * 70)
if sil_hdbscan > sil_kmeans:
    print(f"KESIMPULAN: HDBSCAN unggul pada Silhouette "
          f"({sil_hdbscan:.4f} > {sil_kmeans:.4f}) di array UMAP yang identik,")
    print("            sekaligus mampu mendeteksi noise — sesuatu yang tidak dimiliki K-Means.")
else:
    print(f"CATATAN: Silhouette K-Means ({sil_kmeans:.4f}) >= HDBSCAN ({sil_hdbscan:.4f}). "
          "Periksa kembali sebelum menarik kesimpulan.")
print("=" * 70)
